# GPT-SoVITS 自己声音微调 · Colab T4 全流程

这是重写版，替换掉原来那个把「WebUI 路线」和「手动绕过路线」混在一起的 Notebook。
原版为什么必然失败，见最后一节。

## 怎么用

1. 第 1 步的 4 个变量改成你自己的路径。
2. 从上往下按顺序跑。第 2 步会**中断一次**，这是设计如此（下面解释）。
3. 每一步都有验收输出，看到 OK / 绿色才继续。

## 为什么第 2 步要重启内核

第 2 步装的是 **conda**（condacolab）。它要把 Colab 自带的那个 Python 整个换掉，
而 Python 没法在运行中替换自己，所以必须重启内核才生效。
所以会弹出 `Your session crashed for an unknown reason` 让你重连 —— **这不是报错**。

重启的副作用是内存和 `/content` 全被清空，所以：

- Google Drive 必须**重新挂载**（第 3a 步会再挂一次）
- 之前 `git clone` 下来的代码没了，要在第 3b 步重新拉

**第 2 步只做一次**；重连之后从第 3 步往下跑就行。

## 1. 基本设置（只改这 4 行）

后面的单元都会自动读这些变量。

In [ ]:
%%writefile /content/gsv_config.sh
# ============ 只需要改这 4 行 ============
export EXP="voice_liuyue"                              # 实验名：英文/数字/下划线，不能有空格
export VER="v2Pro"                                     # 版本：v2Pro / v2ProPlus / v2 / v1
export SLICED="/content/drive/MyDrive/Colab/output"    # 0b 切分后的音频目录（在 Drive 上）
export SRC_AUDIO="/content/drive/MyDrive/Colab/raw"    # 原始音频目录（只有要重做 0b 才用到）

# ============ T4 训练参数（跑不动就调小） ============
export S2_BATCH=4          # SoVITS 批大小，爆显存就改 2
export S2_EPOCHS=8         # SoVITS 轮数
export S2_SAVE_EVERY=4     # 每几轮存一次权重
export S1_BATCH=6          # GPT 批大小
export S1_EPOCHS=15        # GPT 轮数

# ============ 以下是自动推导，不用改 ============
export ROOT="/content/GPT-SoVITS"
export OPT="$ROOT/logs/$EXP"
export ASR_OUT="$ROOT/output/asr_opt"
export LIST_NAME="$(basename "$SLICED")"
export LIST="$ASR_OUT/$LIST_NAME.list"
export BERT_DIR="$ROOT/GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large"
export SSL_DIR="$ROOT/GPT_SoVITS/pretrained_models/chinese-hubert-base"
export SV_PATH="$ROOT/GPT_SoVITS/pretrained_models/sv/pretrained_eres2netv2w24s4ep4.ckpt"

# 关键：仓库脚本都写 from tools.xxx / from text.xxx / import utils，
# 都假定「仓库根」和「GPT_SoVITS」在 sys.path 上。不加这两项，0c 会报 No module named tools。
export PYTHONPATH="$ROOT:$ROOT/GPT_SoVITS${PYTHONPATH:+:$PYTHONPATH}"

# conda 环境的 lib 必须排在 /usr/local/lib（conda base）前面。
# 否则动态加载器会拿到 base 里那个旧 libstdc++，conda-forge 的库（libjxl 等）
# 会因为找不到 GLIBCXX_3.4.30 而加载失败 —— 这就是 torchcodec 那份报错的来源。
if [ -n "$CONDA_PREFIX" ] && [ -d "$CONDA_PREFIX/lib" ]; then
  LD_LIBRARY_PATH="$CONDA_PREFIX/lib:$LD_LIBRARY_PATH"
fi

# cuDNN / cuBLAS 等 .so 装在 pip 的 nvidia/* 包里，动态加载器默认搜不到，补进 LD_LIBRARY_PATH
SP_NV="$(python -c "import site; print(site.getsitepackages()[0])" 2>/dev/null)/nvidia"
if [ -d "$SP_NV" ]; then
  for _d in "$SP_NV"/*/lib; do
    if [ -d "$_d" ]; then LD_LIBRARY_PATH="$_d:$LD_LIBRARY_PATH"; fi
  done
fi
export LD_LIBRARY_PATH

# 从仓库自己的 config.py 读该版本对应的预训练权重
if [ -f /content/gsv_paths.py ]; then
  eval "$(python /content/gsv_paths.py)"
  if [ -z "$S2G" ] || [ -z "$S1" ]; then
    echo ""
    echo "!! 读不到预训练权重路径（S2G / S1 是空的）。单独跑下面这行看报错："
    echo "   source activate GPTSoVITS && cd /content/GPT-SoVITS && python /content/gsv_paths.py"
    echo ""
  fi
else
  echo "!! /content/gsv_paths.py 不存在 —— 你还没运行第 7 步里那个 %%writefile /content/gsv_paths.py 单元"
fi
echo "EXP=$EXP"
echo "VER=$VER"
echo "LIST=$LIST"
echo "PYTHONPATH=$PYTHONPATH"
echo "S2G=$S2G"
echo "S1=$S1"

## 2. 安装 conda（**会重启，正常**）

跑完会弹「session crashed」，点重新连接。重连后**不要再跑这个单元**，从第 3 步继续。

> 如果 `/content/GPT-SoVITS` 已经存在、conda 也装好了，可以跳过。

In [ ]:
%pip install -q condacolab
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")

## 3. 重启之后：重挂 Drive + 拉代码 + 装依赖

三个单元按顺序跑，第 3c 个大概 10~20 分钟。

In [ ]:
# 3a. 重新挂载 Drive（重启把挂载弄没了）
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
cd /content
if [ ! -d /content/GPT-SoVITS ]; then
  git clone --depth 1 https://github.com/RVC-Boss/GPT-SoVITS.git
fi
cd /content/GPT-SoVITS
mkdir -p GPT_weights SoVITS_weights
conda env list | grep -qw GPTSoVITS || conda create -n GPTSoVITS python=3.10 -y
source activate GPTSoVITS
pip install -q ipykernel
echo "环境就绪: $(which python)"

### 3c 为什么会报 opencc 错误（重要）

你的日志断在这里：

```
[ERROR]:  Pip install failed:
  Building wheel for opencc (pyproject.toml): started
  error: subprocess-exited-with-error
  × Building wheel for opencc (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  ERROR: Failed building wheel for opencc
```

原因：`requirements.txt` 的**第 1 行**是 `--no-binary=opencc`，它强制 pip **从源码编译** opencc。
而 opencc 的 C++ 源码在 Colab 上编译不过（静默退出码 1）。但 PyPI 上其实有现成的预编译包
（`opencc-1.4.2-cp310-cp310-manylinux2014_x86_64.whl`），只是被这一行挡住了。

所以 3c 加了两行补丁：

1. `sed -i "/^--no-binary/d" requirements.txt` —— 去掉强制源码编译
2. `pip install --only-binary=opencc opencc==1.4.2` —— 直接装官方 wheel

> **这是官方 `install.sh` 自身的 bug，与你的改动无关**；直接跑官方 Colab notebook 也会断在同一处。
> 要还原仓库文件：`cd /content/GPT-SoVITS && git checkout requirements.txt`

> `bash: /usr/local/lib/libtinfo.so.6: no version information available` 是 condacolab 的噪声，可以忽略。

---

顺便，3c 现在还多打了一个补丁：装完之后会把 `torchaudio` 按 `torch` 的版本从 cu126 源强制对齐一次。

原因：`install.sh` 的 CUDA 126 分支只装了 `torch` 和 `torchcodec`，**没有装 `torchaudio`**；而 `requirements.txt`
里的 `torchaudio` 是不锁版本的，pip 会从 PyPI 拉最新版（按 CUDA 13 编译），和 cu126 的 torch 不匹配 ——
这就是 6c 看到的 `OSError: libcudart.so.13`。详见下一节 3e。


In [ ]:
%%bash
# 3c. 装依赖 + 下载预训练权重（10~20 分钟）
#
# 官方 install.sh 现在有三个坑，这段脚本全都填掉：
#   坑 1: requirements.txt 第一行 --no-binary=opencc 强制源码编译 opencc，Colab 上编译不过
#   坑 2: install.sh 的 CUDA 126 分支只装 torch + torchcodec，不装 torchaudio；
#         requirements.txt 里的 torchaudio 不锁版本 -> pip 会装最新版。
#         而 torchaudio >= 2.9 的 torchaudio.load 只能走 torchcodec，torchcodec 在 Colab 上加载不了
#         -> 6c 报 libcudart.so.13，功能测试报 Could not load libtorchcodec
#   坑 3: 所以 torch / torchaudio 必须成对锁到 cu126 源上的 2.8.0（最后一版不依赖 torchcodec 的）

cd /content/GPT-SoVITS
source activate GPTSoVITS
IDX="https://download.pytorch.org/whl/cu126"
PAIR="2.8.0"

# ---- 补丁 1：删掉强制源码编译 opencc 的那一行 ----
sed -i "/^--no-binary/d" requirements.txt
head -2 requirements.txt

# ---- 补丁 2：直接装官方预编译 wheel ----
pip install -q --only-binary=opencc "opencc==1.4.2"
pip show opencc | head -2

# ---- 补丁 3：先把 torch / torchaudio 按 cu126 装成同源同版本 ----
pip install -q "torch==$PAIR" "torchaudio==$PAIR" --index-url "$IDX"

# ---- 走官方安装脚本（已下好的 575MB 预训练权重和 G2PWModel 会自动跳过）----
# 下载慢/失败就换成: bash install.sh --device CU126 --source HF-Mirror
bash install.sh --device CU126 --source HF \
  || echo "!! install.sh 中途失败了。pip 依赖和预训练权重一般已经装好，先往下走，第 4 步体检会告诉我们缺什么。"

# ---- 补丁 4：install.sh 里的 requirements.txt 可能把 torchaudio 顶成最新版，装完再锁一次 ----
pip install -q --force-reinstall --no-deps "torch==$PAIR" "torchaudio==$PAIR" --index-url "$IDX"
# ---- 补丁 5：torchcodec 只有 torchaudio>=2.9 才用，仓库里没有任何代码 import 它，卸掉 ----
pip uninstall -y -q torchcodec 2>/dev/null || true

echo ""
echo "---- 装完的结果（两行都要能打印出来）----"
python -c "import torch;print('torch     ',torch.__version__)"
python -c "import torchaudio;print('torchaudio',torchaudio.__version__)"


### 3d. 修官方仓库的 import 路径 bug（**必跑**）

你在 6c 遇到的报错：

```
File "/content/GPT-SoVITS/tools/asr/fasterwhisper_asr.py", line 12, in <module>
    from tools.asr.config import get_models
ModuleNotFoundError: No module named tools
```

原因是 Python 的一个硬规则：`python -s tools/asr/xxx.py` 只会把**脚本所在目录**（`tools/asr`）放进 `sys.path`，
**不包含当前工作目录**。而仓库里几乎所有脚本都写的是 `from tools.xxx` / `from text.xxx` / `import utils`，
都假定「仓库根目录」和「GPT_SoVITS 目录」在 `sys.path` 上。

顺带说明：这也正是你最早在 WebUI 里点 **0c ASR 显示成功却没有生成文件** 的真正原因 ——
WebUI 用 `Popen(cmd, shell=True)` 拉起这个脚本，报错只打到 `webui.py` 那个 cell 的输出流里，
界面照样显示完成。**这是上游仓库的 bug，不是你的问题。**

下面这个单元往 conda 环境的 `site-packages` 里放一个 `.pth` 文件，把这两个目录永久加进 `sys.path`，
这样无论是我们的单元还是 WebUI 拉起的子进程都能正常 `import tools`。

In [ ]:
%%bash
source activate GPTSoVITS
cd /content/GPT-SoVITS

SP=$(python -c "import site; print(site.getsitepackages()[0])")
: > "$SP/gsv_path.pth"
echo "/content/GPT-SoVITS" >> "$SP/gsv_path.pth"
echo "/content/GPT-SoVITS/GPT_SoVITS" >> "$SP/gsv_path.pth"
echo "写入 $SP/gsv_path.pth :"
cat "$SP/gsv_path.pth"

echo ""
echo "---- 1) 核心包能否导入 ----"
for m in tools text module utils config; do
  if python -c "import $m" 2>/dev/null; then echo "OK   $m"; else echo "FAIL $m"; fi
done

echo ""
echo "---- 2) 模拟「脚本模式」验证 ASR 依赖（模拟 6c 的调用方式）----"
python -s -c "import sys; sys.path.pop(0); from tools.asr.config import get_models" 2>/dev/null \
  && echo "OK   tools.asr 可导入（这就是 6c 之前失败的那一步）" \
  || echo "FAIL tools.asr 仍无法导入"

### 3e. 修 torch / torchaudio（**必跑**）

6c 之后你连续撞了两个错，其实是同一个根因的两张脸：

```
OSError: libcudart.so.13: cannot open shared object file
OSError: Could not load this library: .../torchaudio/lib/_torchaudio.abi3.so
```
```
RuntimeError: Could not load libtorchcodec.
  OSError: /usr/local/lib/libstdc++.so.6: version `GLIBCXX_3.4.30' not found
```

**根因（拆开官方 wheel 源码确认过，不是猜的）**

- `install.sh` 的 CUDA 126 分支只装 `torch` + `torchcodec`，**不装 `torchaudio`**；`requirements.txt` 里的
  `torchaudio` 又不锁版本 → pip 装了最新版，而且和 cu126 的 torch 本来就不同源。
- **`torchaudio` 从 2.9 起，`load` 被改成只能走 torchcodec**：`torchaudio/__init__.py` 直接调用
  `load_with_torchcodec`，没有任何回退路径。2.8 及以前还是老的多后端加载器
  （`torchaudio/_backend/`，ffmpeg / sox / soundfile 自动挑一个）。
- torchcodec 要求特定 FFmpeg 版本 + 新版 libstdc++，在 Colab 的 conda 环境里必然加载失败。
- 另外 cu126 源上 torchaudio 最高只到 2.11，装不出和 torch 2.14 同版的。

**修法**：把 `torch` 和 `torchaudio` 一起锁到 cu126 源上**同源同版本**的 `2.8.0`
（2.8.0 的 wheel 元数据就是 `Requires-Dist: torch==2.8.0`，必须成对），并卸掉 `torchcodec`
（仓库里没有任何代码 `import torchcodec`，它只是 `install.sh` 顺手装的）。

> 全仓库只有两个地方用 torchaudio：第 7 步的 `2-get-sv.py`（v2Pro 专用）和第 10 步的推理。
> 训练（`s2_train.py` / `s1_train.py`）完全不用它，所以这一步不影响别的东西。

> 这一格最后会跑**功能测试**：真的存/读一个 wav + 重采样 + CUDA 运算 —— 光 `import` 成功不算数，
> 因为之前正好就是"import 全 OK、一 `load` 就炸"。


In [ ]:
%%bash
# 3e. 修 torch / torchaudio（**6c 和功能测试的两个错，根因是同一个**）
#
# 你撞到的两个错：
#   OSError: libcudart.so.13: cannot open shared object file
#   OSError: Could not load this library: .../torchaudio/lib/_torchaudio.abi3.so
#   RuntimeError: Could not load libtorchcodec.  (下面跟着 GLIBCXX_3.4.30 / libavutil 缺失)
#
# 根因（拆开官方 wheel 源码确认过，不是猜的）：
#   * install.sh 的 CUDA 126 分支只装 torch + torchcodec，不装 torchaudio；
#     requirements.txt 里的 torchaudio 又不锁版本 -> pip 装了最新版。
#   * torchaudio 从 2.9 起，torchaudio.load 被改成【只能走 torchcodec】：
#     torchaudio/__init__.py 里直接调用 load_with_torchcodec，没有回退路径。
#     torchaudio 2.8 及以前还是老的多后端加载器（_backend/，ffmpeg / sox / soundfile 自动挑）。
#   * torchcodec 要求特定 FFmpeg 版本 + 新版 libstdc++，在 Colab 的 conda 环境里必然加载失败。
#   * 而且 cu126 源上的 torchaudio 最高只到 2.11，跟你的 torch 2.14 本来就不同源。
#
# 所以修法不是"补系统库"，而是把 torch 和 torchaudio 一起锁到 cu126 源上【同源同版本】的 2.8.0：
# 2.8.0 的 wheel 元数据是 Requires-Dist: torch==2.8.0（必须成对），这一代的 load 走 soundfile，
# 完全不需要 torchcodec。

source activate GPTSoVITS
cd /content/GPT-SoVITS
IDX="https://download.pytorch.org/whl/cu126"
PAIR="2.8.0"

echo "---- 换之前 ----"
python -c "import torch;print('torch     ',torch.__version__)" 2>/dev/null || echo "torch 读取失败"
python -c "import torchaudio;print('torchaudio',torchaudio.__version__)" 2>/dev/null || echo "torchaudio 读取失败"

echo ""
echo "---- 装 torch==$PAIR + torchaudio==$PAIR（cu126，同源）----"
pip install -q --force-reinstall --no-deps "torch==$PAIR" "torchaudio==$PAIR" --index-url "$IDX"

echo "---- 卸掉 torchcodec（仓库里没有任何代码 import 它）----"
pip uninstall -y -q torchcodec 2>/dev/null || true

echo ""
echo "---- 功能测试：真的存/读 wav + 重采样 + CUDA 运算（光 import 成功不算数）----"
python - <<'PY'
import wave
import numpy as np
import torch
import torchaudio

p = "/tmp/_gsv_ta_test.wav"
with wave.open(p, "wb") as f:
    f.setnchannels(1)
    f.setsampwidth(2)
    f.setframerate(16000)
    f.writeframes((np.random.randn(16000) * 3000).astype("<i2").tobytes())

try:
    print("  可用后端:", torchaudio.list_audio_backends())
except Exception as e:
    print("  后端列表读取失败:", e)

w, sr = torchaudio.load(p)
print("  torchaudio.load     OK:", tuple(w.shape), sr)
print("  Resample(sr -> 32k) OK:", tuple(torchaudio.transforms.Resample(sr, 32000)(w).shape))
if torch.cuda.is_available():
    a = torch.randn(64, 64, device="cuda", dtype=torch.float16)
    print("  CUDA matmul         OK:", tuple((a @ a).shape))
print("  版本:", torch.__version__, "/", torchaudio.__version__)
PY

echo ""
echo "---- 顺便确认 6c 要用的两个包没被换坏 ----"
for m in funasr faster_whisper; do
  python -c "import $m" 2>/dev/null && echo "OK   $m" || echo "FAIL $m"
done
echo ""
echo "上面功能测试 4~5 行都打印出来 = 修好了，可以进第 7 步。"


## 4. 体检：环境 + 预训练权重

两个单元都必须全绿才能继续。

In [ ]:
%%writefile /content/gsv_doctor.py
import importlib
import torch

for m in ["torch","torchaudio","pytorch_lightning","transformers","gradio","numpy",
          "faster_whisper","funasr","modelscope","ctranslate2","librosa","peft"]:
    try:
        mod = importlib.import_module(m)
        print("%-20s %s" % (m, getattr(mod, "__version__", "?")))
    except Exception as e:
        print("%-20s FAIL: %s" % (m, e))
print()
print("要求: pytorch_lightning >= 2.4 / transformers >= 4.51 / gradio < 5")
print("cuda:", torch.cuda.is_available(), torch.version.cuda)
print("gpu :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无")

In [ ]:
%%bash
source activate GPTSoVITS
source /content/gsv_config.sh
cd "$ROOT"
python /content/gsv_doctor.py

In [ ]:
%%bash
source activate GPTSoVITS
source /content/gsv_config.sh
for f in "$S2G" "$S2D" "$S1" "$S2CFG" "$S1CFG" "$BERT_DIR" "$SSL_DIR" "$SV_PATH"; do
  if [ -e "$f" ]; then echo "OK   $f"; else echo "MISS $f"; fi
done
echo "有 MISS 就先补下载再往下走: bash install.sh --device CU126 --source HF-Mirror"

## 5. 0b 语音切片（你已经做过，可跳过）

WebUI 的 0b 调用的就是这个脚本。默认参数：threshold=-34, min_length=4000, min_interval=300,
hop_size=10, max_sil_kept=500, max=0.9, alpha=0.25。
切片文件名形如 `原始文件名_起始帧_结束帧.wav`。

In [ ]:
%%bash
set -e
source activate GPTSoVITS
source /content/gsv_config.sh
mkdir -p "$SLICED"
cd "$ROOT"
python -s tools/slice_audio.py "$SRC_AUDIO" "$SLICED" -34 4000 300 10 500 0.9 0.25 0 1
echo "切片数量 = $(ls "$SLICED" | wc -l)"

## 6. 0c 语音识别：生成标注文件 .list

用官方脚本，**报错会直接显示在这里**。

> 前置条件：**3d**（`.pth` 路径修复）和 **3e**（torchaudio 对齐）都必须已经跑过。
> 本单元开头会做一次依赖自检，`tools / torch / torchaudio / funasr / faster_whisper` 有一项 FAIL，就先回去重跑那一步。
> 同时会打印切片目录和 wav 数量，先确认输入不是空的。
> 如果你之前跑过 3d 还是报这个错，说明 `.pth` 没生效，把 3d 的输出贴给我。

验收：行数 ≈ 切片数，每行格式是 `绝对路径|说话人名|语言|文本`。
**如果是 0 行，说明 ASR 其实失败了**，往上翻看 traceback。

> 补充坑：`fasterwhisper_asr.py` 的单条转写失败会被 `except` 吞掉（`traceback.print_exc()` 之后不写入该行），
> 所以全部失败时会生成一个**空 list 并打印「ASR 任务完成」**。看到"完成"不等于成功，必须看行数。

> **数据量警告（很重要）**：GPT-SoVITS 微调官方建议 **5 分钟 ~ 1 小时** 的干净人声。
> 目前切片是 17 段、每段 2~7 秒，合计约 1.5 分钟 —— 能跑完，但大概率学不像、音色发飘。
> 想效果好，先往 `SLICED` 目录里补音频（同一个人、无背景音、无 BGM、无混响），再重跑本格。

> 如果日志里出现 `[Errno 21] Is a directory: .../output/bk`，说明切片目录里混了个子文件夹，
> 它会被跳过（无害），但会让 list 少一行。把那个文件夹挪出 `SLICED` 再重跑即可。


In [ ]:
%%bash
# 6c. 语音识别（官方脚本），失败原因直接打印在这里，不会甩一堆 IPython traceback
source activate GPTSoVITS
source /content/gsv_config.sh
cd "$ROOT"

echo "---- 0) 输入检查 ----"
if [ ! -d "$SLICED" ]; then
  echo "!! 切片目录不存在: $SLICED"
  echo "   改 /content/gsv_config.sh 里的 SLICED，或者先做第 5 步切片"
  exit 0
fi
NWAV=$(ls "$SLICED"/*.wav 2>/dev/null | wc -l)
echo "切片目录 = $SLICED"
echo "wav 数量 = $NWAV"
if [ "$NWAV" -eq 0 ]; then
  echo "!! 目录里没有 wav —— 这样跑出来就是 0 行的 list，先解决切片"
  exit 0
fi

echo ""
echo "---- 1) 依赖自检（有 FAIL 就先回去重跑 3d / 3e）----"
for m in tools torch torchaudio funasr faster_whisper; do
  python -c "import $m" 2>/dev/null && echo "OK   $m" || echo "FAIL $m"
done

echo ""
echo "---- 2) 跑 ASR（第一次会自动下载 large-v3 模型，约 1.5GB，慢是正常的）----"
mkdir -p "$ASR_OUT"
python -s tools/asr/fasterwhisper_asr.py -i "$SLICED" -o "$ASR_OUT" -s large-v3 -l zh -p float16
echo "ASR 退出码 = $?"

echo ""
echo "---- 3) 结果 ----"
if [ -s "$LIST" ]; then
  NL=$(wc -l < "$LIST")
  echo "OK  list = $LIST"
  echo "    行数 = $NL   （wav 数量 = $NWAV）"
  echo "    前 3 行："
  head -3 "$LIST"
  if [ "$NL" -lt "$NWAV" ]; then
    echo ""
    echo "    注意：行数比 wav 数少。常见原因是切片目录里混进了子文件夹或非 wav 文件，"
    echo "          日志里会看到 [Errno 21] Is a directory —— 那一项被跳过，本身无害。"
    echo "          把 $SLICED 里除 .wav 之外的东西挪出去，再重跑本格即可。"
  fi
else
  echo "!! list 不存在或为空 —— ASR 失败了，往上翻 traceback"
fi


## 7. 1A / 1B / 1C 数据集预处理（等价 WebUI 的 1A）

生成训练真正需要的 6 样东西：

| 产物 | 说明 |
|---|---|
| `2-name2text.txt` | 4 列：name / phones / word2ph / norm_text |
| `3-bert/*.pt` | BERT 特征 |
| `4-cnhubert/*.pt` | HuBERT 特征 |
| `5-wav32k/*.wav` | 32k 重采样音频 |
| `7-sv_cn/*.pt` | 说话人向量（仅 v2Pro / v2ProPlus） |
| `6-name2semantic.tsv` | 真实 vq 语义 token，首行是表头 |

**这 6 样必须由官方脚本生成。手写或伪造会让训练在加载数据的第一秒就崩。**

In [ ]:
%%writefile /content/gsv_paths.py
# 从仓库的 config.py 读当前版本对应的预训练权重路径，输出成 shell export
import os
import sys

ROOT = "/content/GPT-SoVITS"
os.chdir(ROOT)
sys.path.insert(0, ROOT)
from config import pretrained_sovits_name, pretrained_gpt_name

ver = os.environ["VER"]
s2g = os.path.join(ROOT, pretrained_sovits_name[ver])
s1 = os.path.join(ROOT, pretrained_gpt_name[ver])
if ver in ("v2Pro", "v2ProPlus"):
    s2cfg = os.path.join(ROOT, "GPT_SoVITS/configs/s2" + ver + ".json")
else:
    s2cfg = os.path.join(ROOT, "GPT_SoVITS/configs/s2.json")
s1cfg = os.path.join(ROOT, "GPT_SoVITS/configs/s1longer.yaml" if ver == "v1" else "GPT_SoVITS/configs/s1longer-v2.yaml")

print("export S2G=" + chr(34) + s2g + chr(34))
print("export S2D=" + chr(34) + s2g.replace("s2G", "s2D") + chr(34))
print("export S1=" + chr(34) + s1 + chr(34))
print("export S2CFG=" + chr(34) + s2cfg + chr(34))
print("export S1CFG=" + chr(34) + s1cfg + chr(34))

> 上面是 `%%writefile`，只把脚本写到磁盘，不执行。下面那个才执行。

In [ ]:
%%writefile /content/gsv_prep.py
# 等价 WebUI 的 1A(1A/1B/1C)，但每一步的报错都直接打印出来
import os
import subprocess
import sys

ROOT = os.environ["ROOT"]
EXP = os.environ["EXP"]
VER = os.environ["VER"]
LIST = os.environ["LIST"]
OPT = os.environ["OPT"]
os.makedirs(OPT, exist_ok=True)

BASE = dict(inp_text=LIST, inp_wav_dir="", exp_name=EXP, opt_dir=OPT,
            i_part="0", all_parts="1", _CUDA_VISIBLE_DEVICES="0",
            is_half="True", version=VER,
            bert_pretrained_dir=os.environ["BERT_DIR"],
            cnhubert_base_dir=os.environ["SSL_DIR"],
            sv_path=os.environ["SV_PATH"],
            pretrained_s2G=os.environ["S2G"],
            s2config_path=os.environ["S2CFG"])

def run(script):
    env = os.environ.copy()
    env.update(BASE)
    print("")
    print("=" * 72)
    print(">>> " + script)
    print("=" * 72)
    sys.stdout.flush()
    p = subprocess.run([sys.executable, "-s", script], cwd=ROOT, env=env)
    if p.returncode != 0:
        raise SystemExit("报错终止: " + script + " 退出码 " + str(p.returncode) + "，往上翻看 traceback")
    print("<<< 完成: " + script)

def merge(tmp, out, header=""):
    if not os.path.exists(tmp):
        raise SystemExit("没有生成 " + tmp + " —— 上一个脚本其实失败了，往上翻看 traceback")
    body = open(tmp, encoding="utf8").read().strip("\n")
    with open(out, "w", encoding="utf8") as f:
        if header:
            f.write(header + "\n")
        f.write(body + "\n")
    os.remove(tmp)
    print("[合并] " + tmp + "  ->  " + out)

if not os.path.exists(LIST):
    raise SystemExit("找不到 list 文件: " + LIST + "（先做第 5 / 6 步）")
lines = [x for x in open(LIST, encoding="utf8").read().strip("\n").split("\n") if x.strip()]
if len(lines) == 0:
    raise SystemExit("list 是空文件: " + LIST + " —— ASR 其实失败了，去看第 6 步的报错")
N = len(lines)
print("[list] %d 行" % N)

run("GPT_SoVITS/prepare_datasets/1-get-text.py")
merge(OPT + "/2-name2text-0.txt", OPT + "/2-name2text.txt")

run("GPT_SoVITS/prepare_datasets/2-get-hubert-wav32k.py")

if VER in ("v2Pro", "v2ProPlus"):
    run("GPT_SoVITS/prepare_datasets/2-get-sv.py")

run("GPT_SoVITS/prepare_datasets/3-get-semantic.py")
merge(OPT + "/6-name2semantic-0.tsv", OPT + "/6-name2semantic.tsv", header="item_name\tsemantic_audio")

print("")
print("================ 产物验收 ================")
ok = True
dirs = ["3-bert", "4-cnhubert", "5-wav32k"]
if VER in ("v2Pro", "v2ProPlus"):
    dirs.append("7-sv_cn")
for d in dirs:
    p = os.path.join(OPT, d)
    n = len(os.listdir(p)) if os.path.isdir(p) else -1
    print("  %-12s %s 个" % (d, n))
    if n < N:
        ok = False
for f in ["2-name2text.txt", "6-name2semantic.tsv"]:
    p = os.path.join(OPT, f)
    txt = open(p, encoding="utf8").read().strip("\n") if os.path.exists(p) else ""
    n = len([x for x in txt.split("\n") if x.strip()])
    print("  %-20s %s 行" % (f, n))
    if n < N:
        ok = False
first = open(os.path.join(OPT, "2-name2text.txt"), encoding="utf8").readline().rstrip("\n")
print("  2-name2text 首行列数 = %d (应为 4)" % len(first.split(chr(9))))
if not ok:
    raise SystemExit("产物不完整，先别训练。往上翻看是哪个脚本报错了。")
print("")
print("预处理完成，可以进入第 8 步训练")

In [ ]:
%%bash
source activate GPTSoVITS
source /content/gsv_config.sh
cd "$ROOT"
python /content/gsv_prep.py

## 8. 训练（SoVITS + GPT）

参数名注意：`s2_train.py` 用 `--config`，`s1_train.py` 用 `--config_file`。
原版 1B 手动单元就是在这里传了不存在的 `--config_path`，所以「很快结束、没有任何产物」。

T4 16G 大约 40~90 分钟，取决于切片数量。

In [ ]:
%%writefile /content/gsv_train.py
# 等价 WebUI 的 1B，但错误可见、参数名正确
import json
import os
import subprocess
import sys
import yaml

ROOT = os.environ["ROOT"]
EXP = os.environ["EXP"]
VER = os.environ["VER"]
OPT = os.environ["OPT"]
TEMP = os.path.join(ROOT, "TEMP")
os.makedirs(TEMP, exist_ok=True)

S2_BATCH = int(os.environ.get("S2_BATCH", "4"))
S2_EPOCHS = int(os.environ.get("S2_EPOCHS", "8"))
S2_SAVE = int(os.environ.get("S2_SAVE_EVERY", "4"))
S1_BATCH = int(os.environ.get("S1_BATCH", "6"))
S1_EPOCHS = int(os.environ.get("S1_EPOCHS", "15"))

for f in ["2-name2text.txt", "6-name2semantic.tsv", "4-cnhubert", "5-wav32k"]:
    if not os.path.exists(os.path.join(OPT, f)):
        raise SystemExit("缺 " + f + "，先跑第 7 步")
if VER in ("v2Pro", "v2ProPlus") and not os.path.isdir(os.path.join(OPT, "7-sv_cn")):
    raise SystemExit("缺 7-sv_cn，先跑第 7 步")

# ---------- 权重输出目录（关键：WebUI 会自己建，我们这条无头路线不会） ----------
# webui.py 启动时会执行：for root in SoVITS_weight_root + GPT_weight_root: os.makedirs(root, exist_ok=True)
# 我们这条路线没有这一步。而 s2_train.py 里的 savee() 是【把异常吞掉、只 log 一行】的，
# 所以目录不存在时训练会正常"跑完"、却一个权重文件都不产出 —— 这就是"很快结束、没有产物"的真正机制。
S2_OUT = os.path.join(ROOT, "SoVITS_weights_" + VER)
S1_OUT = os.path.join(ROOT, "GPT_weights_" + VER)
os.makedirs(S2_OUT, exist_ok=True)
os.makedirs(S1_OUT, exist_ok=True)
print("[输出目录] " + S2_OUT + "  |  " + S1_OUT)

# ---------- SoVITS (s2) ----------
os.makedirs(os.path.join(OPT, "logs_s2_" + VER), exist_ok=True)
d = json.load(open(os.environ["S2CFG"], encoding="utf8"))
d["train"].update(dict(
    batch_size=S2_BATCH, epochs=S2_EPOCHS, text_low_lr_rate=0.4,
    grad_ckpt=True, gpu_numbers="0", lora_rank=32,
    pretrained_s2G=os.environ["S2G"], pretrained_s2D=os.environ["S2D"],
    if_save_latest=True, if_save_every_weights=True, save_every_epoch=S2_SAVE))
d["model"]["version"] = VER
d["data"]["exp_dir"] = OPT
d["s2_ckpt_dir"] = OPT
d["save_weight_dir"] = S2_OUT
d["name"] = EXP
d["version"] = VER
p2 = os.path.join(TEMP, "tmp_s2.json")
json.dump(d, open(p2, "w", encoding="utf8"), ensure_ascii=False)
print("[配置] " + p2)

# ---------- GPT (s1) ----------
os.makedirs(os.path.join(OPT, "logs_s1"), exist_ok=True)
s1 = yaml.safe_load(open(os.environ["S1CFG"], encoding="utf8"))
s1["train"].update(dict(batch_size=S1_BATCH, epochs=S1_EPOCHS, save_every_n_epoch=1))
s1["train"].update(dict(if_save_every_weights=True, if_save_latest=True, if_dpo=False,
                        half_weights_save_dir=S1_OUT,
                        exp_name=EXP))
s1["pretrained_s1"] = os.environ["S1"]
s1["train_semantic_path"] = os.path.join(OPT, "6-name2semantic.tsv")
s1["train_phoneme_path"] = os.path.join(OPT, "2-name2text.txt")
s1["output_dir"] = os.path.join(OPT, "logs_s1_" + VER)
p1 = os.path.join(TEMP, "tmp_s1.yaml")
open(p1, "w", encoding="utf8").write(yaml.dump(s1, default_flow_style=False, allow_unicode=True))
print("[配置] " + p1)

env = os.environ.copy()
env.update(_CUDA_VISIBLE_DEVICES="0", hz="25hz")

def run(cmd, label):
    print("")
    print("=" * 72)
    print(">>> " + label)
    print("=" * 72)
    sys.stdout.flush()
    rc = subprocess.run(cmd, cwd=ROOT, env=env).returncode
    if rc != 0:
        raise SystemExit(label + " 失败，退出码 " + str(rc))
    print("<<< 完成: " + label)

run([sys.executable, "-s", "GPT_SoVITS/s2_train.py", "--config", p2], "SoVITS (s2) 训练")
run([sys.executable, "-s", "GPT_SoVITS/s1_train.py", "--config_file", p1], "GPT (s1) 训练")

print("")
print("================ 训练产物 ================")
found = 0
for label, path in [("SoVITS_weights_" + VER, S2_OUT), ("GPT_weights_" + VER, S1_OUT)]:
    files = sorted(os.listdir(path)) if os.path.isdir(path) else []
    found += len(files)
    print(label + "  (" + str(len(files)) + " 个):")
    for n in files:
        print("   " + n)
if found == 0:
    raise SystemExit(
        "!! 训练没有任何产物。往上翻 SoVITS / GPT 的日志。"
        "最常见原因是权重输出目录没建好 —— savee() 会静默失败，训练照样报完成。"
    )
print("")
print("训练完成，接着跑第 9 步备份到 Drive。")

In [ ]:
%%bash
source activate GPTSoVITS
source /content/gsv_config.sh
cd "$ROOT"
python /content/gsv_train.py

## 9. 备份到 Google Drive

Colab 随时会断，训练完第一件事就是备份。

In [ ]:
%%bash
# 必须先 activate：下面 gsv_config.sh 会用 python 跑 /content/gsv_paths.py（要 import torch），
# 不激活就会用系统 python，报 ModuleNotFoundError: No module named 'torch'。
source activate GPTSoVITS
source /content/gsv_config.sh
D="/content/drive/MyDrive/Colab/GPT_SoVITS_Models_Backup"
mkdir -p "$D"
cp -v "$ROOT/SoVITS_weights_$VER"/* "$D"/ 2>/dev/null || true
cp -v "$ROOT/GPT_weights_$VER"/* "$D"/ 2>/dev/null || true
# 顺手备份以后复现需要的小文件
cp -v "$LIST" "$D"/ 2>/dev/null || true
cp -v "$ROOT/TEMP/tmp_s2.json" "$ROOT/TEMP/tmp_s1.yaml" "$D"/ 2>/dev/null || true
echo "---- $D ----"
ls -la "$D"


## 10. 用训练出来的模型推理

### 先解释你贴的那个 WebUI 报错

```
File "gradio/routes.py", line 432, in main
    return templates.TemplateResponse(
File "starlette/templating.py", line 148, in TemplateResponse
    template = self.get_template(name)
TypeError: unhashable type: 'dict'
```

后面那句 `ValueError: When localhost is not accessible, a shareable link must be created.`
**是它的连带结果，不是第二个问题。** 链路是这样的：

1. 仓库 `requirements.txt` 写的是 `gradio<5`，所以只能装到 **gradio 4.44.1**；
   同一份文件里的 `fastapi[standard]>=0.115.2` **没有上限**，pip 会装到最新的 fastapi，
   顺带把 `starlette` 装成 **1.x**。
2. gradio 4.44.1 里是**老签名**：`templates.TemplateResponse("frontend/index.html", {...})`。
   starlette 0.41 还留着兼容分支（打个 DeprecationWarning 就放行），
   **starlette 1.x 把兼容分支删了**，第一个参数被当成 `Request`，
   模板名就变成了那个 dict → jinja2 拿它当缓存 key → `unhashable type: 'dict'`。
3. 首页 `/` 直接 500。
4. gradio 启动前会先自检：`gradio/networking.py` 里 `url_ok()` 用
   `httpx.head(http://127.0.0.1:9874/)`，拿到 200 才认为 localhost 可达。
   首页 500 → 自检失败 → 抛 `When localhost is not accessible...`。

所以这是**版本不兼容**：改端口、挂代理、换 share 都没用。10a 把版本降回去就行。

> 顺带一提：官方 notebook 最后一格是 `export is_share=True && python webui.py`，
> 你没设 `is_share`。就算页面不崩，Colab 里也得有链接才能打开 ——
> 因为 `%%bash` 起的子进程没有 IPython 调用栈，gradio 检测不到自己在 Colab
> （`gradio/utils.py: colab_check()` 是翻调用栈找 `google.colab` 的），
> 所以它**不会**自动生成 Colab 端口代理，只会告诉你 localhost 起好了。

先跑 **10-0 恢复权重**（新会话里 `/content` 是空的），然后 10a 修 WebUI 的版本问题、10b 启动 WebUI，
只想听效果就直接跳到 10c。

### 10a. 先修版本（只在你要开 WebUI 时才需要）

下面这一格把 `fastapi` / `starlette` 退回到 gradio 4.44 时代的组合，
跑完会自己断言验证，最后一步还会 `import gradio.routes`。

再往下：10b 是启动 WebUI（嵌进 notebook，不生成公开链接），
10c 是**完全不用 WebUI** 的直接合成 —— 只想听效果的话，直接跳到 10c。


In [ ]:
%%bash
# 10-0. 新会话必跑：把 Drive 备份里的权重拷回本地目录
# （Colab 一断 /content 就清空，训练产物只活在 Drive 备份里）
source activate GPTSoVITS
source /content/gsv_config.sh

B="/content/drive/MyDrive/Colab/GPT_SoVITS_Models_Backup"
if [ ! -d "$B" ]; then
  echo "!! 找不到备份目录 $B"
  echo "   先跑第 3a 步挂载 Drive；备份路径不一样的话改上面这行 B=..."
  exit 0
fi

mkdir -p "$ROOT/SoVITS_weights_$VER" "$ROOT/GPT_weights_$VER"
cp -v "$B"/*.pth  "$ROOT/SoVITS_weights_$VER"/ 2>/dev/null || true
cp -v "$B"/*.ckpt "$ROOT/GPT_weights_$VER"/ 2>/dev/null || true

echo ""
echo "---- 权重现在长这样 ----"
ls -l "$ROOT/SoVITS_weights_$VER" "$ROOT/GPT_weights_$VER" 2>/dev/null

echo ""
echo "---- 推理还要的两样东西（都在 /content 上，新会话会丢）----"
for p in "$OPT/5-wav32k" "$LIST"; do
  if [ -e "$p" ]; then echo "有   $p"; else echo "缺   $p"; fi
done
echo ""
echo '缺了也别慌：5-wav32k 只是训练时的中间产物，推理用 Drive 上 0b 的原始切片当参考音频完全没问题。'
echo '10c 会自动去 $SLICED 挑一段，并用 output.list 按文件名查原文；实在查不到才需要 REF_TEXT= 手打。'


In [ ]:
%%bash
# 10a. 修 gradio / starlette 版本不兼容（一次性，必须跑）
source activate GPTSoVITS
cd /content/GPT-SoVITS

echo "---- 修之前 ----"
python -c "
import importlib
for m in ['gradio', 'fastapi', 'starlette', 'jinja2']:
    try:
        print('%-10s %s' % (m, importlib.import_module(m).__version__))
    except Exception as e:
        print('%-10s ERR %s' % (m, e))
"

# fastapi 0.115.14 的约束是 starlette>=0.40,<0.47；
# 0.41.3 还保留着 TemplateResponse 的老签名兼容分支。
# --no-deps：这几个包的依赖（anyio / pydantic / jinja2 / typing_extensions）本来就装好了，
# 不带 --no-deps 反而可能被 fastapi-cli 之类的东西拖进依赖冲突里。
pip install -q --no-deps "fastapi==0.115.14" "starlette==0.41.3"

echo "---- 修之后 ----"
python -c "
import inspect
import gradio, fastapi, starlette, jinja2
print('gradio    ', gradio.__version__)
print('fastapi   ', fastapi.__version__)
print('starlette ', starlette.__version__)
print('jinja2    ', jinja2.__version__)
assert starlette.__version__.startswith('0.41'), 'starlette 没降下来，检查 pip 有没有报错'
from starlette.templating import Jinja2Templates
src = inspect.getsource(Jinja2Templates.TemplateResponse)
assert 'isinstance(args[0], str)' in src, '这个 starlette 已经不认 gradio 的老调用了，降到更低版本'
print('老签名兼容分支: 有')
import gradio.routes
print('gradio.routes 导入 OK')
"


### 10b. 启动 WebUI

`webui.py` 第 4 行硬编码 `os.environ["version"] = version = "v2Pro"`，和你训练的版本一致，不用改。

下面这格把 WebUI 放在**后台**跑，等端口起来、确认首页返回 200，再用 Colab 的端口代理
**嵌进 notebook** 里。这样只用你自己的 Colab 会话能访问，**不会生成公开链接**。

想停掉 WebUI：跑一次 `!pkill -f "python webui.py"`。

> 如果你更想要一个能在手机/别的电脑上打开的公开链接（gradio.live，谁拿到都能用），
> 把 10b 换成这两格：
>
> ```bash
> %%bash
> source activate GPTSoVITS
> source /content/gsv_config.sh
> cd /content/GPT-SoVITS
> export is_share=True
> python webui.py
> ```
>
> 然后从输出里抄那个 `https://xxxx.gradio.live` 链接。这是**公开**地址，记得用完关掉。


In [ ]:
# 10b. 后台启动 WebUI，并嵌到 notebook 里
import os
import socket
import subprocess
import time

from google.colab import output

PORT = 9874  # 仓库 config.py 里的 webui_port_main
LOG = "/content/webui.log"

# 从 gsv_config.sh 读出你设置的实验名 / 版本，方便下面打印权重文件名
cfg = {}
if os.path.exists("/content/gsv_config.sh"):
    for line in open("/content/gsv_config.sh", encoding="utf8"):
        s = line.strip()
        for key in ("EXP", "VER"):
            if s.startswith("export %s=" % key):
                cfg[key] = s[len("export %s=" % key):].strip().strip('"').strip("'")
EXP = cfg.get("EXP", "<你的实验名>")
VER = cfg.get("VER", "v2Pro")
print("实验名 EXP = %s，版本 VER = %s" % (EXP, VER))

subprocess.run(["pkill", "-f", "python webui.py"], capture_output=True)
time.sleep(1)
if os.path.exists(LOG):
    os.remove(LOG)

cmd = (
    "source activate GPTSoVITS && source /content/gsv_config.sh "
    "&& cd /content/GPT-SoVITS && exec python webui.py"
)
proc = subprocess.Popen(["bash", "-lc", cmd], stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("WebUI 进程号:", proc.pid)
print('想停掉 WebUI，在一个新的代码格里运行:  !pkill -f "python webui.py" ')

ok = False
for _ in range(90):  # 最多等 3 分钟
    if proc.poll() is not None:
        print("进程已经退出，退出码", proc.returncode)
        break
    with socket.socket() as s:
        if s.connect_ex(("127.0.0.1", PORT)) == 0:
            ok = True
            break
    time.sleep(2)
time.sleep(5)

print("---- webui.log 末尾 ----")
print(open(LOG, encoding="utf8", errors="replace").read()[-3000:])
print("---- 首页自检（要 200；前面所有报错都会在这里现形）----")
r = subprocess.run(
    ["curl", "-s", "-o", "/dev/null", "-w", "GET / -> HTTP %{http_code}",
     "http://127.0.0.1:%d/" % PORT],
    capture_output=True, text=True,
)
out = (r.stdout or "") + (r.stderr or "")
print(out)
code_ok = "200" in out

if ok and code_ok:
    print("")
    print("WebUI 就绪。下面这个 iframe 就是它。在「1C-推理」页里选：")
    print("  SoVITS 权重 : SoVITS_weights_%s/%s_e8_s176.pth" % (VER, EXP))
    print("  GPT 权重    : GPT_weights_%s/%s-e15.ckpt" % (VER, EXP))
    print("  参考音频    : logs/%s/5-wav32k/ 里挑一段 3~10 秒干净人声" % EXP)
    print("  参考原文    : 上面那个 wav 在 output.list 里对应那行的第 4 段（要逐字一致）")
    print("")
    print("想列出「文件名 + 原文」：  !grep -F 文件名 /content/GPT-SoVITS/output/asr_opt/*.list")
    print("")
    # 放到最后渲染：这样 iframe 出现在文字下面
    try:
        output.serve_kernel_port_as_iframe(PORT, height=820)
    except TypeError:
        output.serve_kernel_port_as_iframe(PORT)
else:
    print("WebUI 没起来 / 首页不是 200，别看 iframe，先按上面的日志和自检结果排查")


### 10c. 不用 WebUI，直接合成（推荐先跑这个）

训练的最终目的是听效果，WebUI 只是外壳。这一节**不依赖 10a**（gradio 的模板渲染坏了，但推理代码本身没事），
理论上可以单独跑。脚本直接调用
`GPT_SoVITS/inference_webui.py` 里的 `change_gpt_weights / change_sovits_weights / get_tts_wav`
（**和 WebUI 里「合成语音」按钮完全同一份代码**），产出 wav 后用 `IPython.display.Audio` 内嵌播放。

- 参考音频默认先看 `logs/$EXP/5-wav32k/`，没有（比如新会话里 `/content` 被清空）就退到 `$SLICED`
  —— 也就是你在 Drive 上放 0b 切片那个目录。**5-wav32k 不是推理必需品**，它只是训练的中间产物；
  原始切片一样能当参考音频。脚本会自动挑一段 3~10 秒的。
- 参考文本从 `output.list` 里按**文件名**取（两边 basename 完全一致），保证「参考音频 ↔ 参考文本」逐字一致
  （写错会明显跑调）。`output.list` 不在默认位置也没事，脚本会在 `output/asr_opt`、`$SLICED`、
  `$SLICED` 的上一级、`/content`、仓库根目录里自己找一圈；实在没有才要 `REF_TEXT=` 手打。
- 权重默认取 `SoVITS_weights_$VER` / `GPT_weights_$VER` 里**最新**的那份，也可以用环境变量指定。
- 想换某一轮的 GPT 权重（比如少过拟合一点的 `e10`），取消注释 `GPT_W=` 那行即可。

> 提醒：官方自己的 `GPT_SoVITS/inference_cli.py` 是坏的 —— 它直接调用
> `change_sovits_weights(...)` 而不迭代。那个函数是**生成器**，不迭代就等于**根本没加载 SoVITS 权重**。
> 我们的脚本用 `for _ in change_sovits_weights(SOV_W): pass` 绕开这个坑。


In [ ]:
%%writefile /content/gsv_tts.py
# 不启动 WebUI，直接在命令行合成语音（用的是 WebUI 里同一份推理代码）
#
# 云端:  source activate GPTSoVITS && source /content/gsv_config.sh && cd "$ROOT"
#        TEXT="要念的句子" python /content/gsv_tts.py
#
# 本机 (Windows): 建议直接用同一目录下的 gsv_tts_local.py（纯命令行参数，不用环境变量）
# 或者按 10d 节里的环境变量写法:
#        $env:ROOT="D:\GPT-SoVITS"; $env:VER="v2Pro"
#        $env:REF_AUDIO="D:\ref\a.wav"; $env:REF_TEXT="参考音频里原封不动说的那句话"
#        $env:TEXT="要念的句子"; python gsv_tts.py
#
# 只有 ROOT / VER 是必须的，其它（OPT / LIST）找不到会自动降级并给提示。
import os
import sys
import time

ROOT = os.environ.get("ROOT")
if not ROOT or not os.path.isdir(ROOT):
    raise SystemExit("环境变量 ROOT 没设或不是目录 —— 它要指向 GPT-SoVITS 仓库根目录")

VER = os.environ.get("VER", "v2Pro")
OPT = os.environ.get("OPT", "")        # logs/<实验名>，只用来找默认参考音频（可选）
LIST = os.environ.get("LIST", "")      # output/asr_opt/*.list，只用来查参考音频原文（可选）
SLICED = os.environ.get("SLICED", "")  # 0b 的切片目录（在 Drive 上），参考音频的兜底来源
OUT = os.path.abspath(os.environ.get("OUT_DIR") or os.path.join(ROOT, "tts_out"))
os.makedirs(OUT, exist_ok=True)

# 必须在 import inference_webui 之前设好：它开头就读 os.environ.get("version", "v2")
os.environ["version"] = VER

# 和 webui.py 写 users.pth 时一样的顺序，缺一个都不行（from text / from module / from sv / import bigvgan）
for sub in ["", "GPT_SoVITS/BigVGAN", "tools", "tools/asr", "GPT_SoVITS", "tools/uvr5"]:
    p = os.path.join(ROOT, sub)
    if os.path.isdir(p) and p not in sys.path:
        sys.path.append(p)


def newest(d, ext):
    if not os.path.isdir(d):
        return None
    fs = [os.path.join(d, f) for f in os.listdir(d) if f.endswith(ext)]
    return max(fs, key=os.path.getmtime) if fs else None


GPT_W = os.environ.get("GPT_W") or newest(os.path.join(ROOT, "GPT_weights_" + VER), ".ckpt")
SOV_W = os.environ.get("SOVITS_W") or newest(os.path.join(ROOT, "SoVITS_weights_" + VER), ".pth")
for p in (GPT_W, SOV_W):
    if not p or not os.path.exists(p):
        raise SystemExit(
            "找不到权重: GPT=%s  SoVITS=%s\n"
            "（要放在 %s\GPT_weights_%s 和 %s\SoVITS_weights_%s 里，或用 GPT_W= / SOVITS_W= 指过去）"
            % (GPT_W, SOV_W, ROOT, VER, ROOT, VER)
        )

# ---- 参考文本表: output.list 每行是 "路径|说话人|语种|文本"，官方下游脚本都只取 basename ----
# 新会话里 /content 是空的，而这个文件经常被你手动挪到 Drive 或别处，所以这里自己找一圈。
def find_list():
    base = os.path.basename(SLICED.rstrip("/\\")) if SLICED else ""
    dirs = []
    for d in [os.path.join(ROOT, "output", "asr_opt"), os.path.join(ROOT, "output"),
              SLICED, os.path.dirname(SLICED) if SLICED else "", "/content", ROOT]:
        if d and os.path.isdir(d) and d not in dirs:
            dirs.append(d)
    if LIST:
        if os.path.exists(LIST):
            return LIST
        for d in dirs:
            if base and os.path.exists(os.path.join(d, base + ".list")):
                return os.path.join(d, base + ".list")
    for d in dirs:
        for f in sorted(os.listdir(d)):
            if f.endswith(".list"):
                return os.path.join(d, f)
    return ""


name2text = {}
LIST = find_list()
if LIST:
    print("[list] 参考原文从这里查:", LIST)
    with open(LIST, encoding="utf-8-sig") as f:
        for line in f.read().strip("\n").split("\n"):
            part = line.split("|", 3)
            if len(part) == 4:
                name2text[os.path.basename(part[0])] = part[3].strip()
else:
    print("!! 没找到任何 *.list，参考文本只能用 REF_TEXT= 手动给")

import soundfile as sf


def pick_ref(d):
    """在目录里挑一段 3~10 秒的 wav；有对应原文的优先（参考文本必须逐字一致）"""
    if not os.path.isdir(d):
        return None
    cands = []
    for f in sorted(os.listdir(d)):
        if not f.endswith(".wav"):
            continue
        try:
            dur = sf.info(os.path.join(d, f)).duration
        except Exception:
            continue
        if 3.0 <= dur <= 10.0:
            cands.append((0 if f in name2text else 1, abs(dur - 7.0), os.path.join(d, f)))
    return min(cands)[2] if cands else None


REF = os.environ.get("REF_AUDIO")
if REF and not os.path.exists(REF):
    raise SystemExit("REF_AUDIO 指的文件不存在: %s" % REF)
if not REF:
    # 5-wav32k 只是训练的中间产物，不是推理必需品；用 Drive 上 0b 的原始切片当参考音频完全没问题
    for d in [(os.path.join(OPT, "5-wav32k") if OPT else ""), SLICED]:
        if d and os.path.isdir(d):
            REF = pick_ref(d)
            if REF:
                print("[参考音频] 从 %s 里自动挑了一段" % d)
                break
if not REF:
    raise SystemExit("没有可用的参考音频 —— 用 REF_AUDIO=<一段 3~10 秒的干净人声> 指定")

REF_TEXT = os.environ.get("REF_TEXT") or name2text.get(os.path.basename(REF))
if not REF_TEXT:
    raise SystemExit(
        "拿不到 %s 的原文 —— 用 REF_TEXT='那段音频里原封不动说的话' 手动给" % os.path.basename(REF)
    )

TEXT = os.environ.get("TEXT") or "".join(sys.argv[1:]) or "今天天气不错，我们一起出去走走吧。"

print("GPT 权重   :", GPT_W)
print("SoVITS 权重:", SOV_W)
print("参考音频   :", REF)
print("参考文本   :", REF_TEXT)
print("合成文本   :", TEXT)
print("输出目录   :", OUT)
print("")

import GPT_SoVITS.inference_webui as iw
from GPT_SoVITS.inference_webui import change_gpt_weights, change_sovits_weights, get_tts_wav


def lang_key(code):
    # dict_language 的 key 是按系统语言 i18n 过的，按 value 反查最稳
    for k, v in iw.dict_language.items():
        if v == code:
            return k
    raise SystemExit("这个版本不支持 %s，支持: %s" % (code, list(iw.dict_language.values())))


PROMPT_CODE = os.environ.get("REF_LANG", "all_zh")
TEXT_CODE = os.environ.get("TGT_LANG", "all_zh")

t0 = time.time()
change_gpt_weights(GPT_W)
print("GPT 权重加载完成 %.1fs" % (time.time() - t0))

t0 = time.time()
for _ in change_sovits_weights(SOV_W, lang_key(PROMPT_CODE), lang_key(TEXT_CODE)):
    pass  # 生成器，不迭代 = 没加载；且必须带上两个语种，否则官方代码会 UnboundLocalError
print("SoVITS 权重加载完成 %.1fs" % (time.time() - t0))

t0 = time.time()
results = list(
    get_tts_wav(
        ref_wav_path=REF,
        prompt_text=REF_TEXT,
        prompt_language=lang_key(PROMPT_CODE),
        text=TEXT,
        text_language=lang_key(TEXT_CODE),
        top_k=int(os.environ.get("TOP_K", "15")),
        top_p=float(os.environ.get("TOP_P", "1.0")),
        temperature=float(os.environ.get("TEMPERATURE", "1.0")),
        speed=float(os.environ.get("SPEED", "1.0")),
        how_to_cut=iw.i18n(os.environ.get("CUT", "按标点符号切")),
    )
)
sr, wav = results[-1]
path = os.path.join(OUT, "tts_%d.wav" % int(time.time()))
sf.write(path, wav, sr)
print("")
print("合成完成 %.1fs" % (time.time() - t0))
print("输出文件:", path, "（%.2f 秒音频）" % (len(wav) / float(sr)))


In [ ]:
%%bash
source activate GPTSoVITS
source /content/gsv_config.sh
cd "$ROOT"

# ========= 想改的只有这几行 =========
export TEXT="今天天气不错，我们一起出去走走吧。"
# export REF_AUDIO="$OPT/5-wav32k/voice_liuyue_origin2-enhanced-v2.wav_0000777600_0000916160.wav"
# export GPT_W="$ROOT/GPT_weights_$VER/voice_liuyue-e10.ckpt"
# export SOVITS_W="$ROOT/SoVITS_weights_$VER/voice_liuyue_e8_s176.pth"
# export TEMPERATURE=1.0      # 越大越随机、越小越平，1.0 是 WebUI 默认
# export SPEED=1.0
# ===================================

python /content/gsv_tts.py


In [ ]:
# 10c-3. 播放刚生成的音频，并顺手存一份到 Drive
import glob
import os
import shutil

from IPython.display import Audio, display

files = []
for pat in ("/content/out/tts_*.wav", "/content/GPT-SoVITS/tts_out/tts_*.wav"):
    files += glob.glob(pat)
files.sort(key=os.path.getmtime)

if not files:
    print("还没有产物，先跑上面那个单元")
else:
    print("最新文件:", files[-1])
    display(Audio(files[-1]))
    try:
        dst = "/content/drive/MyDrive/Colab/gsv_out"
        os.makedirs(dst, exist_ok=True)
        shutil.copy(files[-1], dst)
        print("已复制到", dst)
    except Exception as e:
        print("存 Drive 失败（Drive 没挂？）:", e)


### 10d. 想在自己电脑上跑（纯 CPU / 没显卡）

可以，而且**不用去找整合包**。用官方仓库装 `--Device CPU`，那就是和你训练时同一份代码，
v2Pro 权重才保证能加载。网上很多整合包停在 v2 或 v3 时代，`config.py` 里根本没有 `v2Pro`
这一项，权重丢进去也选不到 —— 下载前先在包里搜一下 `v2Pro`，搜不到就别下。

```pwsh
# 1) 装好 Miniconda / Anaconda 之后
conda create -n GPTSoVits python=3.10 -y
conda activate GPTSoVits

# 2) 拉同一份代码（不要用旧的整合包）
git clone https://github.com/RVC-Boss/GPT-SoVITS.git
cd GPT-SoVITS

# 3) 装依赖（CPU 版 torch + 预训练权重；国内用 HF-Mirror）
pwsh -File install.ps1 -Device CPU -Source HF-Mirror
```

**装完先自检 —— 这个安装脚本每个平台都有同一个坑**：`install.ps1` 只装 `torch torchcodec`，
而 `requirements.txt` 里是**裸的 `torchaudio`**，pip 会顺手把 CPU 版 torch 换成 CUDA 版
（白白多下 2.5GB），而新版 torchaudio 的 `load` 只能走 torchcodec —— 就是你之前在 6c 撞到的
`_torchaudio.abi3.so` / `libtorchcodec` 那类报错。

```pwsh
python -c "import torch, torchaudio; print(torch.__version__, torchaudio.__version__)"
```

如果 torchaudio 是 2.9 以上、或者 torch 被换成了 `+cu126` 之类，就锁回 2.8.0 CPU 版
（和云端 3e 步同一个版本，已验证可用）：

```pwsh
pip uninstall -y torch torchaudio torchcodec
pip install --index-url https://download.pytorch.org/whl/cpu torch==2.8.0 torchaudio==2.8.0
```

> 如果安装卡在 `Building wheel for opencc` 失败：和云端同一个原因 ——
> `requirements.txt` 第一行的 `--no-binary=opencc` 强制源码编译，而 PyPI 上明明有
> `opencc-1.4.2-cp310-win_amd64.whl`。把那一行删掉再重跑：
>
> ```pwsh
> (Get-Content requirements.txt) | Where-Object { $_ -notmatch '^--no-binary' } | Set-Content requirements.txt
> pwsh -File install.ps1 -Device CPU -Source HF-Mirror
> ```

把权重放进去（文件名就按你训练出来的）：

- `GPT_weights_v2Pro\voice_liuyue-e15.ckpt`
- `SoVITS_weights_v2Pro\voice_liuyue_e8_s176.pth`

然后两条路随便挑：

- **WebUI**：双击 `go-webui.bat`（或 `pwsh -File go-webui.ps1`）→ 打开 `1-GPT-SoVITS-TTS / 1C-inference`。
- **命令行（推荐）**：用 `gsv_tts_local.py`（纯命令行参数版，不依赖任何环境变量）；
  它和 `gsv_tts.py` 是同一套逻辑，但权重、参考音频、原文、批量句子全部走命令行参数：

```pwsh
# 权重不给就用 GPT_weights_v2Pro / SoVITS_weights_v2Pro 里最新的那份
python gsv_tts_local.py --root "D:/GPT-SoVITS" `
    --ref "D:/ref/你的5秒参考.wav" `
    --ref-text "那段参考音频里原封不动说的话" `
    --text "今天天气不错，我们一起出去走走吧。"

# 更省事：给它一个切片目录 + output.list，参考音频和参考原文都自动搞定
python gsv_tts_local.py --root "D:/GPT-SoVITS" `
    --ref-dir "D:/colab_slices" --list "D:/colab_slices/output.list" `
    --text "今天天气不错"

# 批量合成（一行一句），试「哪一轮 GPT 权重最稳」时特别省事
python gsv_tts_local.py --root "D:/GPT-SoVITS" --ref-dir "D:/colab_slices" ^
    --list "D:/colab_slices/output.list" --text-file 句子.txt ^
    --gpt "D:/GPT-SoVITS/GPT_weights_v2Pro/voice_liuyue-e10.ckpt"
```

（`gsv_tts.py` 那套环境变量写法也还能用，只是不如上面直观。）

**CPU 模式有个流传很广的过时说法，别再照着做**：

- 旧版 `config.py` 里确实有 `device = "cuda"` 和 `is_half = True` 两条让你手改；
  **当前版本已经没有这两行了**，改成自动判断 —— `config.py` 的 `get_device_dtype_sm()`
  在没有 CUDA 时直接 `return cpu, torch.float32`，`is_half = any(dtype == torch.float16 ...)`。
  所以**纯 CPU 的机器什么都不用改**，一启动就是 CPU + float32。
- 只有「机器上有显卡、但你想强制用 CPU」才需要动手：启动前设 `CUDA_VISIBLE_DEVICES=""`
  （`torch.cuda.is_available()` 会变成 False，自动落到 CPU）。**别去改 `config.py`。**
- 速度预期：CPU 比显卡慢一个数量级左右，一句 10 秒的话大概几十秒；慢是正常的，不是卡死。
  内存建议 ≥ 8GB，磁盘预留 ~10GB。

> 本地 CPU 这条路的意义是**无限次试**（云端额度有限）。只想听一次效果的话，直接用 10c 就行。


## 附录：原 Notebook 为什么必然失败

| # | 问题 | 证据 |
|---|---|---|
| 1 | 手动 1B 传了不存在的参数 | `s1_train.py` 只接受 `-c/--config_file`，原单元传的是 `--config_path` → argparse 立即退出 |
| 2 | 手动 1A 伪造语义 token | 写的是 `名字\t0 1 2 3`，真实值由 `3-get-semantic.py` 用 vq 模型抽出上千个 token |
| 3 | 手动 1A 的文本表只有 2 列 | 官方是 4 列 `name / phones / word2ph / norm_text`（`1-get-text.py`） |
| 4 | 四个特征目录一个都没生成 | `3-bert` / `4-cnhubert` / `5-wav32k` / `7-sv_cn`；`data_utils.py` 里有 `os.listdir(self.path4)` 和 `assert os.path.exists(self.path7)` |
| 5 | 环境被降级破坏 | 官方要求 `pytorch-lightning>=2.4`、`transformers>=4.51,<5`、`gradio<5`；原单元装了 `pl==1.9.5`、`transformers==4.39.3`、升级了 gradio、把 torch 降到 2.1.2 |
| 6 | `torch.utils._pytree` 假模块 | 是 PL 1.9.5 与 torch 2.x 不兼容的产物，只掩盖报错，要删掉 |
| 7 | WebUI 版本被硬编码成 v2Pro | `webui.py` 第 4 行 `os.environ["version"] = version = "v2Pro"`；`python webui.py v2` 里的 v2 其实是**语言**参数 |
| 8 | 权重缺失不报错 | WebUI 会把不存在的预训练路径静默改成空字符串，1A 抛 `FileNotFoundError` 但界面仍显示完成 |
| 9 | ASR 失败也会显示成功 | `fasterwhisper_asr.py` 的 `except` 吞掉每一条的异常，`output` 为空时照样写出空 list 并打印「ASR 任务完成」 |
| 10 | `install.sh` 的 CUDA 126 分支不装 `torchaudio` | 源码是 `pip install torch torchcodec --index-url .../cu126`，而 `requirements.txt` 里是裸的 `torchaudio` |
| 11 | `torchaudio >= 2.9` 的 `load` 只能走 torchcodec | 2.9.0 的 wheel 里 `def load(` 只定义在 `__init__.py`（转调 `load_with_torchcodec`）；2.8.0 才有完整的 `torchaudio/_backend/` 多后端加载器 |
| 12 | 训练产物目录没人建（无头路线的致命点） | `webui.py` 里有 `for root in SoVITS_weight_root + GPT_weight_root: os.makedirs(root, exist_ok=True)`，脚本路线没有 |
| 13 | `savee()` 把保存异常吞掉 | `process_ckpt.py` 里 `except: return traceback.format_exc()`，`s2_train.py` 只把它 `logger.info` 出来 → 目录不存在 = 训练"完成"但零产物 |
| 14 | `python -s <脚本>` 不把工作目录加进 `sys.path` | `python -s tools/asr/x.py` 只加 `tools/asr`；所有 `from tools.xxx` / `import utils` 都会 `ModuleNotFoundError`（第 3d 步用 `.pth` 修掉） |
| 15 | 中文 ASR 的文本其实由 FunASR 产出 | `fasterwhisper_asr.py` 里 `if info.language in ["zh","yue"]: text = only_asr(...)`，所以 funasr / torchaudio 坏掉会让 0c 直接死 |
| 16 | `gradio<5` 和没有上限的 fastapi 装出致命组合 | `requirements.txt` 同时有 `gradio<5` 和 `fastapi[standard]>=0.115.2`；gradio 4.44.1 用老签名 `TemplateResponse(name, {...})`，starlette 1.x 删了兼容分支 → 首页 500 → 启动自检报 `When localhost is not accessible` |
| 17 | 官方 `inference_cli.py` 不迭代生成器 | `change_sovits_weights()` 是生成器，官方 CLI 里直接调用不 `next()`，等于没加载 SoVITS 权重 |
